# PGVector RAG Query Notebook — Keycloak JWT + RLS Demo

This notebook demonstrates role-scoped document access using PGVector + LangChain, governed by **Keycloak JWTs** — the same identity model used by the MCP server.

- **Jane** (admin token) can query documents from **all** collections
- **John** (user token) can only query documents from the **`user`** collection

The Keycloak JWT determines `app.current_role`, which PostgreSQL RLS policies use to filter rows.

In [1]:
!pip install -r requirements.txt

In [2]:
import base64
import json
import os
from urllib.parse import quote

import requests
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_postgres import PGEngine, PGVectorStore
import psycopg

# PostgreSQL connection — uses the non-superuser 'app' role so RLS is enforced
PG_HOST = os.getenv("PG_HOST", "postgresql.redbank-demo.svc.cluster.local")
PG_PORT = os.getenv("PG_PORT", "5432")
PG_DATABASE = os.getenv("PG_DATABASE", "db")
PG_USER = os.getenv("PG_USER", "app")
PG_PASSWORD = os.getenv("PG_PASSWORD", "app")

# Keycloak configuration
KEYCLOAK_URL = os.getenv("KEYCLOAK_URL", "https://keycloak-keycloak.apps.rosa.redbank-demo2.ij5f.p3.openshiftapps.com")
KEYCLOAK_REALM = os.getenv("KEYCLOAK_REALM", "redbank")
KEYCLOAK_CLIENT = os.getenv("KEYCLOAK_CLIENT", "redbank-mcp")
ADMIN_ROLE = "admin"

# Embedding model — same as the ingestion pipeline
embeddings = HuggingFaceEmbeddings(model_name="nomic-ai/nomic-embed-text-v1.5")

print("Setup complete")

/opt/app-root/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 112/112 [00:00<00:00, 3903.07it/s]


Setup complete


## Keycloak Authentication

Get JWTs for Jane (admin) and John (user), then extract their roles from the token claims — the same way the MCP server does it.

In [3]:
def get_keycloak_token(username: str, password: str) -> str:
    """Get an access token from Keycloak."""
    resp = requests.post(
        f"{KEYCLOAK_URL}/realms/{KEYCLOAK_REALM}/protocol/openid-connect/token",
        data={
            "grant_type": "password",
            "client_id": KEYCLOAK_CLIENT,
            "username": username,
            "password": password,
        },
        verify=False,
    )
    resp.raise_for_status()
    return resp.json()["access_token"]


def extract_role(token: str) -> str:
    """Extract role from JWT claims (same logic as MCP server)."""
    payload = token.split(".")[1]
    payload += "=" * (-len(payload) % 4)  # pad base64
    claims = json.loads(base64.urlsafe_b64decode(payload))
    # Check realm_access.roles for admin role
    realm_roles = claims.get("realm_access", {}).get("roles", [])
    return "admin" if ADMIN_ROLE in realm_roles else "user"


def build_connection_string(role: str) -> str:
    """Build a PostgreSQL connection string that sets app.current_role via options."""
    return (
        f"postgresql+psycopg://{PG_USER}:{PG_PASSWORD}"
        f"@{PG_HOST}:{PG_PORT}/{PG_DATABASE}"
        f"?options={quote(f'-c app.current_role={role}')}"
    )


# Authenticate as Jane (admin) and John (user)
jane_token = get_keycloak_token("jane", "jane123")
jane_role = extract_role(jane_token)
print(f"Jane: role={jane_role}")

john_token = get_keycloak_token("john", "john123")
john_role = extract_role(john_token)
print(f"John: role={john_role}")

Jane: role=admin
John: role=user


/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'keycloak-keycloak.apps.rosa.redbank-demo2.ij5f.p3.openshiftapps.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'keycloak-keycloak.apps.rosa.redbank-demo2.ij5f.p3.openshiftapps.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


## Jane's Query (Admin) — sees all collections

Jane's JWT contains the `admin` realm role, so `app.current_role='admin'` is set on the connection. RLS allows her to see both `admin` and `user` documents.

In [4]:
jane_engine = PGEngine.from_connection_string(url=build_connection_string(jane_role))

jane_store = PGVectorStore.create_sync(
    engine=jane_engine,
    table_name="embeddings",
    embedding_service=embeddings,
    metadata_columns=["collection"],
)

query = "How do I reset my password?"
jane_results = jane_store.similarity_search(query, k=5)

print(f"Jane (role={jane_role}) results ({len(jane_results)} docs):\n")
for i, doc in enumerate(jane_results):
    collection = doc.metadata.get("collection", "unknown")
    print(f"--- [{i+1}] collection={collection} ---")
    print(doc.page_content)
    print()

Jane (role=admin) results (5 docs):

--- [1] collection=user ---
Managing Your Password & Security Settings
RedBank Financial Services - Internal Document
1. Changing Your Password
To change your online banking password: log in to banking.redbank.com or open the RedBank mobile app.
Go to Settings > Security > Change Password. Enter your current password, then enter your new password
twice to confirm. Password requirements: at least 12 characters, must include at least one uppercase letter,
one lowercase letter, one number, and one special character (!@#$%^&*). Cannot reuse any of your last 10
passwords. Your password expires every 90 days and you will be prompted to change it. We recommend
using a password manager to generate and store strong, unique passwords.
2. Setting Up Two-Factor Authentication (2FA)
Two-factor authentication adds an extra layer of security to your account. To enable: go to Settings > Security
> Two-Factor Authentication. Choose your preferred method: Authenticat

## John's Query (User) — sees only `user` collection

John's JWT has no `admin` role, so `app.current_role='user'` is set. RLS restricts him to the `user` collection only — he cannot see admin-only internal procedures.

In [5]:
john_engine = PGEngine.from_connection_string(url=build_connection_string(john_role))

john_store = PGVectorStore.create_sync(
    engine=john_engine,
    table_name="embeddings",
    embedding_service=embeddings,
    metadata_columns=["collection"],
)

john_results = john_store.similarity_search(query, k=5)

print(f"John (role={john_role}) results ({len(john_results)} docs):\n")
for i, doc in enumerate(john_results):
    collection = doc.metadata.get("collection", "unknown")
    print(f"--- [{i+1}] collection={collection} ---")
    print(doc.page_content)
    print()

John (role=user) results (5 docs):

--- [1] collection=user ---
Managing Your Password & Security Settings
RedBank Financial Services - Internal Document
1. Changing Your Password
To change your online banking password: log in to banking.redbank.com or open the RedBank mobile app.
Go to Settings > Security > Change Password. Enter your current password, then enter your new password
twice to confirm. Password requirements: at least 12 characters, must include at least one uppercase letter,
one lowercase letter, one number, and one special character (!@#$%^&*). Cannot reuse any of your last 10
passwords. Your password expires every 90 days and you will be prompted to change it. We recommend
using a password manager to generate and store strong, unique passwords.
2. Setting Up Two-Factor Authentication (2FA)
Two-factor authentication adds an extra layer of security to your account. To enable: go to Settings > Security
> Two-Factor Authentication. Choose your preferred method: Authenticato